# DriftSense: Phase P0 - Environment & Setup Verification

This notebook verifies that the Colab runtime has all dependencies installed, imports the `driftsense` library, confirms no pretrained models/weights exist, and validates configuration loading.

### 1. Setup & Environment
- Clone / pull repo or ensure package is on `PYTHONPATH`
- Install dependencies from `requirements.txt`
- Print Python and core library versions
- Set deterministic seed

In [ ]:
# Mount Google Drive (optional in local / interactive testing, enabled for Colab runs)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
current_dir = Path.cwd()
if (current_dir / 'src').exists():
    sys.path.insert(0, str(current_dir / 'src'))
elif (current_dir.parent / 'src').exists():
    sys.path.insert(0, str(current_dir.parent / 'src'))

# Print runtime environment
import platform
print(f"Python Version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

# Verify key library versions
for lib in ['numpy', 'pandas', 'polars', 'duckdb', 'scipy', 'sklearn', 'lightgbm', 'statsmodels', 'yaml', 'pytest']:
    try:
        mod = __import__(lib)
        print(f"{lib:12s}: {getattr(mod, '__version__', 'available')}")
    except ImportError:
        print(f"{lib:12s}: NOT INSTALLED")

### 2. Configuration & Seed

In [ ]:
import time
start_time = time.time()

import driftsense
from driftsense.utils.seed import set_seed
from driftsense.utils.io import load_config
from driftsense.utils.summary import print_result_summary

# Global scale selection
SCALE = "tiny"  # "tiny" | "small" | "full"

base_cfg = load_config("base.yaml")
scale_cfg = load_config("scale.yaml")
faults_cfg = load_config("faults.yaml")
dirt_cfg = load_config("dirt.yaml")
realism_cfg = load_config("realism.yaml")

SEED = base_cfg.get("seed", 42)
set_seed(SEED)
print(f"Loaded configs successfully. Active Scale: {SCALE}, Seed: {SEED}")

### 3. Run: Smoke Checks & Imports

In [ ]:
# Import all subpackages to confirm clean namespaces
import driftsense.sim
import driftsense.data
import driftsense.forecast
import driftsense.detect
import driftsense.attribute
import driftsense.explain
import driftsense.utils

print(f"DriftSense Version: {driftsense.__version__}")

### 4. Inspect: Environment & Scales

In [ ]:
active_scale_params = scale_cfg[SCALE]
print("Active Scale Parameters:", active_scale_params)
print("Configured Providers:", base_cfg["providers"])
print("Configured Services:", base_cfg["services"])

### 5. Checks: Assertions & Invariant Verification

In [ ]:
checks_status = {}

# Check 1: Invariant - No pretrained imports or references
forbidden = ['from_pretrained', 'transformers', 'torch.hub', 'anthropic', 'openai', 'chronos', 'timesfm']
src_path = Path(driftsense.__file__).parent
violations = []
for py_file in src_path.rglob('*.py'):
    txt = py_file.read_text(encoding='utf-8')
    for pat in forbidden:
        if pat in txt:
            violations.append((str(py_file), pat))

assert len(violations) == 0, f"Pretrained violations found: {violations}"
checks_status["no_pretrained_models"] = "PASS"

# Check 2: Configs presence
assert "tiny" in scale_cfg and "full" in scale_cfg
checks_status["configs_valid"] = "PASS"

# Check 3: Deterministic seeding
assert set_seed(SEED) == SEED
checks_status["seed_deterministic"] = "PASS"

print("All assertions passed successfully:", checks_status)

### 6. Save Artifacts

In [ ]:
# Ensure local / drive artifact directory exists
out_dir = Path("artifacts/p0")
out_dir.mkdir(parents=True, exist_ok=True)
status_file = out_dir / "p0_setup_status.txt"
status_file.write_text("P0 setup and environment verification completed successfully.")
print(f"Artifact written to: {status_file}")

### 7. RESULT SUMMARY

In [ ]:
runtime = time.time() - start_time

summary = {
    "phase": "P0",
    "scale": SCALE,
    "seed": SEED,
    "runtime_sec": runtime,
    "shapes": {"providers": len(base_cfg["providers"]), "services": len(base_cfg["services"])},
    "key_metrics": {"driftsense_version": driftsense.__version__},
    "checks": checks_status,
    "warnings": [],
    "artifacts": [str(status_file)]
}

print_result_summary(summary)